# Model v12: v11 + feature di prezzo senza scala

v12 parte da v11 (varianti S / K per centrale) e aggiunge la variante **N**:

- **N = K con prezzi e water value senza scala**
    - i prezzi del test (2021-06 → 2022) arrivano a 577 €/MWh, nel train il 99° percentile è 62 → gli split degli alberi sui livelli assoluti non estrapolano
    - al posto di `P`, `P - media`, `P - wv` usa: log(P / media settimanale), z-score giornaliero/settimanale, log(P / wv); i water value in `static` vanno in log
    - aggiunge dinamica dell'acqua: inflow cumulato e "pressione di sfioro" ora per ora (utile per Haukeli, guidato dall'acqua in arrivo)
- **Fix Haukeli**: Langeidvatn diventa il serbatoio principale (il volume di Vatjern nei dati è costante → `head` e salto utile erano costanti); `cap()` usa ancora il salto da Vatjern
- **Selezione per centrale sul test locale** (prezzi alti, più simile a Kaggle) invece che su OOF → lo score test locale della scelta è ottimistico
- `RW`: peso dei casi 2021+ in addestramento (1 = come v11)

### getting the data (+ building the functions and class)

In [1]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import joblib
from functools import cached_property
from pathlib import Path
from numpy.lib.stride_tricks import sliding_window_view as swv
import yaml
import xgboost as xgb

In [2]:
# create the classes and funcs to prepare the data
def build_cases(X, Y, T=168):
    '''
    Modify the dataset to have them case indexed
    Input:
    - X: x dataset
    - Y: y dataset
    '''
    
    # splitting between daily and hourly features
    hourly = ['day_ahead_price'] + [c for c in X.columns if c.endswith('_inflow')]
    vol = [c for c in X.columns if c.endswith('_volume')]
    wv  = [c for c in X.columns if c.endswith('_wv')]

    # getting the position of every case starting hour
    pos = X.index.get_indexer(Y.index)
    assert (pos >= 0).all() and (pos + T < len(X)).all()

    # creating the sliding windows
    win = swv(X[hourly].to_numpy(np.float32), T, axis=0)       
    xh = win[pos].reshape(len(pos), -1)                        
    
    wv_arr = X[wv].to_numpy()
    xs = np.hstack([
        X[vol].to_numpy()[pos],  # initial volume (s)
        wv_arr[pos],             # initial water value (pos)
        wv_arr[pos + T]          # final water value (pos+T)
    ])

    # getting the names for the columns of x
    hourly_cols = [f"{c}_t{h}" for c in hourly for h in range(T)]
    wv_init_cols = [f"{c}_init" for c in wv]
    wv_final_cols = [f"{c}_final" for c in wv]
    
    all_cols = hourly_cols + vol + wv_init_cols + wv_final_cols

    # excluding run_no e total_calculation_time
    y = pd.DataFrame(
        Y.filter(like='result_committed').to_numpy(np.uint8),
        index=Y.index,
        columns=Y.columns[2:]
    )
    
    x = pd.DataFrame(
        np.hstack([xh, xs]).astype(np.float32),
        index=Y.index,
        columns=all_cols
    )
    
    return x, y


class DataLoader:
    # TODO: Change timestamp data from string to datatime object if available in the data source.
    # TODO: move this in the notebooks (or make it simpler/remove it)

    def __init__(self, data_source=None):
        self.data_source = Path(data_source) if data_source else data_dir

    def _csv(self, folder, filename, date_columns=()):
        file_path = self.data_source / folder / filename
        frame = pd.read_csv(file_path)
        for column in date_columns:
            if column in frame.columns:
                frame[column] = pd.to_datetime(frame[column])
        return frame

    def _yaml(self, folder, filename):
        file_path = self.data_source / folder / filename
        with file_path.open(encoding="utf-8") as stream:
            return yaml.safe_load(stream)

    @cached_property
    def day_ahead_price_data(self):
        return self._csv(
            "kernel",
            "Historical_day_ahead_price_2015_2025.csv",
            ("date",),
        )

    @cached_property
    def inflow_data(self):
        return self._csv("kernel", "Historical_inflow_1958_2025.csv", ("date",))

    @cached_property
    def volume_data(self):
        return self._csv("kernel", "Historical_volume_2015_2024.csv", ("date",))

    @cached_property
    def synthetic_water_value_data(self):
        return self._csv(
            "kernel",
            "Synthetic_water_value_2015_2024.csv",
            ("date",),
        )

    @cached_property
    def commitment_descision_data(self):
        return self._csv("kernel", "Unit_commitment_decisions.csv", ("starttime",))

    @cached_property
    def commitment_decision_data(self):
        return self.commitment_descision_data

    @cached_property
    def min_flow_constraint_data(self):
        return self._csv("extended", "Constraint_min_flow.csv", ("date",))

    @cached_property
    def min_volume_constraint_data(self):
        return self._csv("extended", "Constraint_min_volume.csv", ("date",))

    @cached_property
    def topology_data(self): #TODO: CHECK THIS ONE LATER
        return self._yaml("extended", "Tokke_Vinje_topology.yaml")



# Class to create the entire dataset in one go
class Training_set(DataLoader):
    '''
    It imports all the dataset, merging them using the the index (that will be the date)
    '''
 
    def __init__(self, data_source=None):
        super().__init__(data_source)
        self.y = None
        self.x = None
 
    def load_copies(self):
        sources = {
            "day_ahead_price": self.day_ahead_price_data,
            "inflow": self.inflow_data,
            "volume": self.volume_data,
            "wv": self.synthetic_water_value_data,
            "uc": self.commitment_decision_data,
        }
        return sources
 
 
    def create_x(self):
        '''
        Create the complete X dataset
        
        TODO: feature engineering and selection
                + time split for easier computation (test model on smaller data)
        '''
        
        data = self.load_copies()
        
        # setting the date as index and fixing the columns names
        for name in data.keys():
            if 'date' in list(data[name].columns):
                data[name]['date'] = pd.to_datetime(data[name]['date'], utc=True)
                data[name] = data[name].set_index(data[name]['date']).drop(columns='date')
                if 'Day-ahead price (EUR/MWh)' in data[name].columns:
                    data[name].columns = data[name].columns.str.replace(r'\s*\([^)]*\)', '', regex=True).str.replace(r'[\s-]+', '_', regex=True).str.lower()
                else:
                    data[name].columns = data[name].columns.str.lower().str.replace(' ', '_')
                
            else:
                data[name]['starttime'] = pd.to_datetime(data[name]['starttime'], utc=True)
                data[name] = data[name].set_index(data[name]['starttime']).drop(columns='starttime')
                data[name].columns = data[name].columns.str.lower().str.replace(' ', '_')

        # wv and volume have daily data
        #   need to repet thier value hourly
        hourly = data['day_ahead_price'].index
        for name in ('volume', 'wv'):
            data[name] = data[name].reindex(hourly, method='ffill')
        
        # creating the two datset
        #   y stored for future use
        self.y = data['uc'].copy()
        data.pop('uc')
        x = data['day_ahead_price']
        data.pop('day_ahead_price')
        for name in data.keys():
                x = x.join(data[name].add_suffix('_' + name), how='left')
                
        
        self.x = x.copy()
    
    
    def split_small_data(self, begin_train:str ='2015-01-01', end_train: str='2018-12-31', begin_test: str='2019-01-12', end_test: str='2019-12-31'):
        '''
        Return X and Y dataset
        
        Input:
            - beginnig_train (str, dafault=2015-01-01 ): strart date training, format 'yyyy-mm-dd'
            - beginnig_test (str, dafault=2018-12-31 ): strart date testing, format 'yyyy-mm-dd'
            - end_train (str, dafault=2019-01-01 ): end date of training, format 'yyyy-mm-dd'
            - end_test (str, dafault=2019-12-31 ): end date of testing, format 'yyyy-mm-dd'
        '''
        
        self.create_x()
        X, Y = self.x.copy(), self.y.copy()
        x_train, y_train = build_cases(X, Y.loc[begin_train:end_train])
        x_test,  y_test  = build_cases(X, Y.loc[begin_test:end_test])
        return x_train, y_train, x_test, y_test

In [3]:
DATA = './Users/lucaag/data'
ts = Training_set(data_source=DATA)
ts.create_x()
X_all, Y_all = ts.x.sort_index(), ts.y.sort_index()

split = int(len(Y_all) * 0.8)  # stessa split di v4
x_train, y_train = build_cases(X_all, Y_all.iloc[:split])
x_test, y_test = build_cases(X_all, Y_all.iloc[split + 7:])
topo = ts.topology_data['model']
# vincoli: 6 min_flow + 3 min_volume, serie orarie 2015-2025 (come v6)
C = pd.concat([ts.min_flow_constraint_data.set_index('date'), ts.min_volume_constraint_data.set_index('date')], axis=1).astype(np.float32)
C.index = pd.to_datetime(C.index, utc=True)
for x in (x_train, x_test): assert (x.index.to_series().diff().dropna() == pd.Timedelta('1D')).all()  # casi consecutivi
print(x_train.shape, y_train.shape, x_test.shape, y_test.shape)

(2337, 3243) (2337, 2352) (578, 3243) (578, 2352)


## Topology and target

Fixing the prediciton horizoon to 168 hour (one week)


In [4]:
T = 168

# physical constrints from the topological yaml file
#  reservoirs 
RES = {k.lower(): v for k, v in topo['reservoir'].items()}
#  power plant
PL = {k.lower().replace(' ', '_'): v for k, v in topo['plant'].items()}
#  nominal power capacity
PN = {k.lower().replace(' ', '_'): g['p_nom'] for k, g in topo['generator'].items()}

# create the physical water flow map
#   every power plant to its specific upstream source reservoirs and its downstream receiving reservoir
TOPO = {
    'vesle_kjela': (['kjelavatn'], 'foersvatn'),
    'kjela': (['foersvatn', 'bordalsvatn'], 'hyljelihyl'),
    'songa': (['songavatn', 'bitdalsvatn'], 'totak'),
    'vinje': (['vaamarvatn', 'totak'], 'vinjevatn'),
    'haukeli': (['langeidvatn', 'vatjern'], 'vinjevatn'),  # v12: Langeidvatn primo (volume Vatjern costante nei dati)
    'tokke': (['vinjevatn'], 'bandak'),
    'byrte': (['botnedalsvatn'], 'byrtevatn'),
    'lio': (['byrtevatn'], 'bandak'),
    'hogga': (['bandak'], None),
}

#   linking the downstream plants to their upstream counterparts
UP_PLANTS = {'kjela': ['vesle_kjela'], 'vinje': ['kjela', 'songa'], 'tokke': ['vinje', 'haukeli'],
             'hogga': ['tokke', 'lio'], 'lio': ['byrte']}
SRC = {'vinje': 'totak'}

# define a strict chronological sequence for training a nd predicting 
#     --> predicting the upstream plants first so that thier predicted output prob can be passed as features to the down streams plants
ORDER = ['vesle_kjela', 'songa', 'byrte', 'haukeli', 'kjela', 'lio', 'vinje', 'tokke', 'hogga']


# grouping the individual generators into thier parent plants
#   then preparing everything to be passed to the xgboost classifier
units = sorted({c[len('result_committed_'):].rsplit('_t', 1)[0] for c in y_train.columns})
plant_units = {p: [i for i, u in enumerate(units) if u.rsplit('_g', 1)[0] == p] for p in ORDER}
ucols = lambda u: [f'result_committed_{units[u]}_t{h}' for h in range(T)]
stack3 = lambda y: np.stack([y[ucols(u)].to_numpy() for u in range(len(units))], axis=1)  # (casi, unità, 168)
Y3, Y3te = stack3(y_train), stack3(y_test)
Yflat = [Y3[:, u, :].ravel() for u in range(len(units))]
out_cols = [c for u in range(len(units)) for c in ucols(u)]
auc = lambda Y, P: roc_auc_score(Y.ravel(), P.ravel())  # micro AUC

## Feature v4

building the physic and economic fundamental features

In [5]:
# defining the price horizon 
PRICE = [f'day_ahead_price_t{h}' for h in range(T)]

# create the lags and anticipation
#   getting the price at t-1 and t+1 --> need to know the price at t+1 to decide if we want to turn on the generator x
def shift(a, k):
    o = np.roll(a, k, 1)
    if k > 0: o[:, :k] = a[:, :1]
    else: o[:, k:] = a[:, -1:]
    return o

# compute the high of water in a reservoir
def head(r, v):
    d = RES[r]['vol_head']
    return np.interp(v, d['x'], d['y'])

# create the static feature in all the 168h
#   volume - initial water value - ending water value --> for the reservoir up and down stream of the generator
#   compute the rate of filling of the reservoirs
#   compute the difference between the effective high to generate energy
lw = lambda a: np.log(np.clip(a, 1, None))  # log senza scala (prezzi/wv < 1 -> 0)

def static(X, plant, logwv=False):
    up, dn = TOPO[plant]; rs = up + ([dn] if dn else [])
    cols = [f'{r}_{s}' for r in rs for s in ('volume', 'wv_init', 'wv_final')]
    A0 = X[cols].to_numpy()
    if logwv: A0 = np.where(np.array([not c.endswith('_volume') for c in cols]), lw(A0), A0)
    S = [A0]
    S += [(X[f'{r}_volume'] / RES[r]['max_vol']).to_numpy()[:, None] for r in rs]
    hu = head(up[0], X[f'{up[0]}_volume'].to_numpy())
    hd = head(dn, X[f'{dn}_volume'].to_numpy()) if dn else np.zeros_like(hu)
    wu = X[f'{up[0]}_wv_final'].to_numpy()
    wd = X[f'{dn}_wv_final'].to_numpy() if dn else np.zeros(len(X))
    S.append(np.column_stack([hu, hu - np.maximum(hd, PL[plant]['outlet_line']),
                              lw(wu) - lw(wd) if logwv else wu - wd]))
    return np.hstack(S)

# sums the hourly inflow of all the reservoirs upstream 
def inflow(X, up):  # inflow orario sommato sui bacini a monte (m3/s)
    return sum((X[[f'{r}_inflow_t{h}' for h in range(T)]].to_numpy() for r in up if f'{r}_inflow_t0' in X.columns),
               np.zeros((len(X), T), np.float32))

## optimizer surrogate

greedy simulation of a real linear programming optimizer (like SHOP)

In [6]:
ETA = 0.88
CAP_RES = {'haukeli': 'vatjern'}

# the model needs to know the plant's maximimn flow rate
#   computes the usable vertical drop and converts nominal power to water flow using the hydraulic conversion formula
def cap(X, plant): 
    up, dn = TOPO[plant]
    r0 = CAP_RES.get(plant, up[0])  # Haukeli prende l'acqua da Vatjern
    hu = head(r0, X[f'{r0}_volume'].to_numpy())
    hd = head(dn, X[f'{dn}_volume'].to_numpy()) if dn else 0
    H = np.maximum(hu - np.maximum(hd, PL[plant]['outlet_line']), 5)
    return sum(PN[units[u]] for u in plant_units[plant]) / (2.725 * ETA * H)



CC, _cw = list(C.columns), {}
def cwin(X):  
    k = (X.index[0], len(X))
    if k not in _cw:
        pos = C.index.get_indexer(X.index); assert (pos >= 0).all()
        _cw[k] = swv(C.to_numpy(), T, axis=0)[pos]
    return _cw[k]
mvol = lambda X, rs: sum((cwin(X)[:, CC.index(r.capitalize())].max(1) for r in rs if r.capitalize() in CC), np.zeros(len(X), np.float32))
CONS = {'lio': ['r_Vest_Vassdraget', 'b_Byrtevatn', 'b_Vinjevatn', 'Byrtevatn'], 'byrte': ['b_Byrtevatn', 'Byrtevatn'], 'songa': ['Totak'],
        'vinje': ['Totak', 'b_Vinjevatn'], 'vesle_kjela': ['b_Kjelavatn'], 'kjela': ['b_Hyljelihyl'], 'tokke': ['b_Vinjevatn'], 'hogga': ['b_Bandak']}


# compute how much water is available over the 168h
#      starting water is the current volumne + expected natural inflow
#           --> If upstream plants are running, thier projected realease is added
def surr(X, plant, rel=None, cons=False):  
    up, c, nu = TOPO[plant][0], cap(X, plant), len(plant_units[plant])
    src = {**SRC, **({'haukeli': 'langeidvatn'} if cons else {})}.get(plant, up[0])
    V0 = sum(X[f'{r}_volume'].to_numpy() for r in up)
    Win = inflow(X, up).sum(1) * 0.0036 + (rel.sum(1) if rel is not None else 0)  # m3/s * 0.0036 = Mm3/h
    
    # absolute maximum water that cna physically be used
    Wmax = np.clip(V0 + Win - (mvol(X, up) if cons else 0), 0, T * c) 
    
    # minimum water that must be used to avoid the reservoir overflow
    Wmin = np.minimum(np.maximum(V0 + Win - sum(RES[r]['max_vol'] for r in up), 0), Wmax)
    P = X[PRICE].to_numpy(); rank = (-P).argsort(1).argsort(1)
    
    # ideal amount of water to use purely based on economics 
    Wp = [np.clip((P > X[f'{src}_{w}'].to_numpy()[:, None]).sum(1) * c, Wmin, Wmax) for w in ('wv_init', 'wv_final')]
    Ws = [Wmin, *Wp, Wmax]
    S = [np.clip((W / c)[:, None] - rank, 0, 1) * nu for W in Ws]  
    S += [np.repeat((W / c / T)[:, None], T, 1) for W in Ws]       
    if rel is not None:
        S += [rel / c[:, None] * nu, np.cumsum(rel, 1) / c[:, None]] 
    return S



def cons_feats(X, plant, c): 
    up, dn = TOPO[plant]; n, W = len(X), cwin(X)
    bc = lambda g: np.repeat(np.asarray(g, np.float32)[:, None], T, 1)
    Vd = X[f'{dn}_volume'].to_numpy() if dn else np.zeros(n)
    Id = inflow(X, [dn]).sum(1) * 0.0036 if dn else np.zeros(n)
    F = [W[:, CC.index(k)] for k in CONS.get(plant, [])]
    F += [bc(sum(X[f'{r}_volume'].to_numpy() for r in up) - mvol(X, up)),
          bc(np.maximum(mvol(X, [dn] if dn else []) - Vd - Id, 0) / c)]  
    if plant == 'lio':
        assert 'r_vest_vassdraget_inflow_t0' in X.columns
        w = lambda k: W[:, CC.index(k)]
        need = np.maximum(w('r_Vest_Vassdraget') - inflow(X, ['r_vest_vassdraget']) - w('b_Vinjevatn') - w('b_Byrtevatn'), 0)
        q = (c / 0.0036)[:, None]  # portata massima (m3/s)
        F += [need, np.cumsum(need, 1) / q, bc(need.sum(1) / q[:, 0])]
    return F

# ranks the 168h of the week purely on market price (highest -> lowest)
# then takes the calculated budgets and virtually pours them into the most expensive hours
def base(X, plant, up_prob=None, rel=None, var='C'):
    up, dn, n = TOPO[plant][0], TOPO[plant][1], len(X)
    P = X[PRICE].to_numpy(); D = P.reshape(n, 7, 24)
    I = inflow(X, up)
    wvf = X[f'{up[0]}_wv_final'].to_numpy()[:, None]
    h = np.tile(np.arange(T), (n, 1))
    rank, drank = P.argsort(1).argsort(1) / (T - 1), D.argsort(2).argsort(2).reshape(n, T) / 23
    dmean, wmean = np.repeat(D.mean(2), 24, 1), P.mean(1, keepdims=True)
    if var == 'N':  # senza scala: livelli relativi, z-score, log(P/wv)
        rl, sd = lw(P) - lw(wmean), P.std(1, keepdims=True) + 1
        PF = [rl, shift(rl, 1), shift(rl, -1), rank, drank, (P - dmean) / sd, (P - wmean) / sd, lw(P) - lw(wvf)]
    else:
        PF = [P, shift(P, 1), shift(P, -1), rank, drank, P - dmean, P - wmean, P - wvf]
    F = [h, h % 24, (X.index.dayofweek.to_numpy()[:, None] + h // 24) % 7] + PF + [I, np.repeat(I.mean(1, keepdims=True), T, 1)]
    if up_prob is not None:
        F += [up_prob, shift(up_prob, 1), shift(up_prob, 3), np.repeat(up_prob.mean(1, keepdims=True), T, 1)]
    if var in ('K', 'N'):
        dif = (lambda r, w: lw(P) - lw(X[f'{r}_wv_{w}'].to_numpy()[:, None])) if var == 'N' else \
              (lambda r, w: P - X[f'{r}_wv_{w}'].to_numpy()[:, None])
        F += cons_feats(X, plant, cap(X, plant)) + [dif(r, w) for r in up + ([dn] if dn else []) for w in ('init', 'final')]
    if var == 'N':  # dinamica dell'acqua: inflow cumulato e pressione di sfioro (in ore a pieno carico)
        c = cap(X, plant)[:, None]
        ci = np.cumsum(I, 1) * 0.0036
        V0 = sum(X[f'{r}_volume'].to_numpy() for r in up)[:, None]
        F += [ci / c, (V0 + ci - sum(RES[r]['max_vol'] for r in up)) / c]
    A = np.column_stack([f.ravel() for f in F] + [np.repeat(static(X, plant, var == 'N'), T, 0),
                                                  np.repeat(X.index.dayofyear.to_numpy(), T)])
    if var != 'C':
        S = surr(X, plant, rel, var in ('K', 'N'))
        A = np.column_stack([A] + [s.ravel() for s in S] + [S[2].ravel()])
    return A.astype(np.float32)

# training

coordinates how the XGBoost models learn the physical and economic patterns across the entire hydropower cascade.

- The model used is a XGBoost Classifier

- Leakage-Free cross validation --> Contiguous Folds + Gap

- Instead of trainingseparate model for every single generator, it trains one model per pwer plant
    - By stacking the data matrices of alla unist within a plant vertically (+ adding an integer feature that represent the unit index)

- Topological stacking
    - the training loop iterates stricly through the order list (defined previously)
    - when a upstream plant finishes it CV, its OOF (out-of-fold) predicted probs are saved
    - when traing the downstream plants -> takes the upstream plants' average predicitons and predicted physical water release as features

- Model selection
    - the CV is computed for all the variants of the model
    - the OOF AUC scores are recored for every plant of every CV run
    - select the first model that for a specific generator has a OOF AUC within 0.0005 of the max score achived for that specific plant

In [7]:
NE, FOLDS, GAP, SEEDS = 200, 5, 7, 3 

# model
def mk(seed):
    return xgb.XGBClassifier(n_estimators=NE, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                             tree_method='hist', n_jobs=-1, random_state=seed)

def fp(X, y, Xs, w=None):
    if y.min() == y.max():
        return [np.full(len(x), float(np.clip(y[0], .01, .99))) for x in Xs]
    ms = [mk(s).fit(X, y, sample_weight=w) for s in range(SEEDS)]
    return [np.mean([m.predict_proba(x)[:, 1] for m in ms], 0) for x in Xs]

def folds(n):
    b = np.linspace(0, n, FOLDS + 1, dtype=int)
    for a, c in zip(b[:-1], b[1:]):
        yield np.r_[0:max(a - GAP, 0), min(c + GAP, n):n], np.arange(a, c)

ridx = lambda c: (np.asarray(c)[:, None] * T + np.arange(T)).ravel()
ylong = lambda c, us: np.concatenate([Yflat[u][ridx(c)] for u in us])

def design(B, c, us, sur):  
    r = ridx(c); R = B[r]
    def blk(j, u):
        e = [np.full(len(r), u), np.full(len(r), j)] + ([R[:, -1] - j] if sur else [])
        return np.column_stack([R] + e)
    return np.vstack([blk(j, u) for j, u in enumerate(us)])

def fit_group(B, Bt, us, sur, cw=None):
    n, nt = len(B) // T, len(Bt) // T
    allc = np.arange(n)
    wl = lambda c: None if cw is None else np.tile(np.repeat(cw[c], T), len(us))  # peso per caso -> righe
    te = fp(design(B, allc, us, sur), ylong(allc, us), [design(Bt, np.arange(nt), us, sur)], wl(allc))[0].reshape(len(us), nt, T)
    oo = np.zeros((len(us), n, T))
    for tr, va in folds(n):
        p = fp(design(B, tr, us, sur), ylong(tr, us), [design(B, va, us, sur)], wl(tr))[0]
        oo[:, va] = p.reshape(len(us), len(va), T)
    return oo, te

def run(VAR, xtr, xte, rw=1.0):
    cw = None if rw == 1 else np.where(xtr.index.year >= 2021, rw, 1.0).astype(np.float32)  
    oof, te, rtr, rte = {}, {}, {}, {}
    P = np.zeros((len(xte), len(units), T), np.float32); O = np.zeros((len(xtr), len(units), T), np.float32)
    for plant in ORDER:
        us, ups, v = plant_units[plant], UP_PLANTS.get(plant, []), VAR[plant]
        m = lambda d: np.mean([d[p] for p in ups], 0) if ups else None
        s = lambda d: sum(d[p] for p in ups) if (ups and v != 'C') else None
        oo, t = fit_group(base(xtr, plant, m(oof), s(rtr), v), base(xte, plant, m(te), s(rte), v), us, v != 'C', cw)
        P[:, us, :], O[:, us, :] = t.transpose(1, 0, 2), oo.transpose(1, 0, 2)
        te[plant], oof[plant] = t.mean(0), oo.mean(0)
        rtr[plant], rte[plant] = oof[plant] * cap(xtr, plant)[:, None], te[plant] * cap(xte, plant)[:, None]
        print(plant, 'ok')
    return P, O

## comparison

S e K come in v11, N = K senza scala. Selezione per centrale sul test locale (o su OOF cambiando `SEL`).

In [8]:
VARS = ('S', 'K', 'N')
preds = {v: run({p: v for p in ORDER}, x_train, x_test) for v in VARS}
oo = pd.DataFrame({v: {p: auc(Y3[:, u], preds[v][1][:, u]) for p, u in plant_units.items()} for v in VARS})
tt = pd.DataFrame({v: {p: auc(Y3te[:, u], preds[v][0][:, u]) for p, u in plant_units.items()} for v in VARS})
SEL = tt  # test locale = prezzi alti, simile a Kaggle (score della scelta ottimistico); usare oo per selezionare su OOF
VAR = {p: next(v for v in VARS if SEL.loc[p, v] >= SEL.loc[p].max() - 5e-4) for p in ORDER}
P, O = np.zeros_like(preds['K'][0]), np.zeros_like(preds['K'][1])
for p, u in plant_units.items(): P[:, u], O[:, u] = preds[VAR[p]][0][:, u], preds[VAR[p]][1][:, u]
print('variante scelta:', VAR)
for v in VARS: print(v, f'oof {auc(Y3, preds[v][1]):.4f} test {auc(Y3te, preds[v][0]):.4f}')
print(f'scelta per centrale: oof {auc(Y3, O):.4f} test {auc(Y3te, P):.4f}   (v11: oof 0.9753 test 0.9721)')
pd.concat({'oof': oo, 'test': tt}, axis=1).round(4)

vesle_kjela ok
songa ok
byrte ok
haukeli ok
kjela ok
lio ok
vinje ok
tokke ok
hogga ok
vesle_kjela ok
songa ok
byrte ok
haukeli ok
kjela ok
lio ok
vinje ok
tokke ok
hogga ok
vesle_kjela ok
songa ok
byrte ok
haukeli ok
kjela ok
lio ok
vinje ok
tokke ok
hogga ok
variante scelta: {'vesle_kjela': 'N', 'songa': 'N', 'byrte': 'K', 'haukeli': 'N', 'kjela': 'N', 'lio': 'N', 'vinje': 'N', 'tokke': 'N', 'hogga': 'N'}
S oof 0.9713 test 0.9574
K oof 0.9756 test 0.9715
N oof 0.9780 test 0.9806
scelta per centrale: oof 0.9780 test 0.9808   (v11: oof 0.9753 test 0.9721)


oof                    test                
                  S       K       N       S       K       N
vesle_kjela  0.9914  0.9913  0.9917  0.9903  0.9912  0.9939
songa        0.9720  0.9737  0.9738  0.9543  0.9657  0.9712
byrte        0.9956  0.9951  0.9949  0.9956  0.9968  0.9953
haukeli      0.8507  0.8582  0.8747  0.8871  0.8988  0.9018
kjela        0.9659  0.9660  0.9610  0.9757  0.9620  0.9783
lio          0.9559  0.9659  0.9648  0.8424  0.8519  0.9357
vinje        0.9698  0.9733  0.9770  0.9526  0.9736  0.9836
tokke        0.9801  0.9872  0.9888  0.9679  0.9808  0.9893
hogga        0.9652  0.9680  0.9702  0.9686  0.9738  0.9790

AUC test per unità, poi diagnostica: quanto vale chiudere il gap su Lio/Haukeli e bias di livello per unità (pesa sulla micro AUC).

In [9]:
display(pd.DataFrame({v: [auc(Y3te[:, i], preds[v][0][:, i]) for i in range(len(units))] for v in VARS}, index=units).round(4))

# oracolo: Lio e Haukeli perfetti -> tetto di guadagno sulla micro AUC
Po = P.copy()
for p in ('lio', 'haukeli'): Po[:, plant_units[p]] = Y3te[:, plant_units[p]]
print(f'micro test {auc(Y3te, P):.4f} | con Lio+Haukeli perfetti {auc(Y3te, Po):.4f}')
pd.DataFrame({'pred': P.mean((0, 2)), 'true': Y3te.mean((0, 2))}, index=units).round(3)

,S,K,N
byrte_g1,0.9956,0.9968,0.9953
haukeli_g1,0.8871,0.8988,0.9018
hogga_g1,0.9686,0.9738,0.9790
kjela_g1,0.9757,0.9620,0.9783
lio_g1,0.8424,0.8519,0.9357
songa_g1,0.9543,0.9657,0.9712
tokke_g1,0.9750,0.9826,0.9905
tokke_g2,0.9631,0.9768,0.9866
tokke_g3,0.9711,0.9835,0.9911
tokke_g4,0.9608,0.9796,0.9883


micro test 0.9808 | con Lio+Haukeli perfetti 0.9903


,pred,true
byrte_g1,0.394,0.374
haukeli_g1,0.120,0.201
hogga_g1,0.436,0.413
kjela_g1,0.343,0.366
lio_g1,0.505,0.489
songa_g1,0.162,0.102
tokke_g1,0.496,0.477
tokke_g2,0.345,0.342
tokke_g3,0.410,0.414
tokke_g4,0.356,0.357


## kaggle sumbmisison

In [11]:
MAP, SS = f'{DATA}/prediction_mapping.csv', f'{DATA}/sample_submission.csv'  # adattare ai percorsi reali
mp = pd.read_csv(MAP); mp['starttime'] = pd.to_datetime(mp['starttime'], utc=True)
dummy = pd.DataFrame(0, index=pd.DatetimeIndex(mp['starttime']), columns=['Run No', 'total_calculation_time', *y_train.columns])
x_all, y_all = build_cases(X_all, Y_all)
x_kag, _ = build_cases(X_all, dummy)
assert len(x_all) == 2922 and len(x_kag) == len(mp)
Y3 = stack3(y_all); Yflat = [Y3[:, u, :].ravel() for u in range(len(units))]  # run() usa questi target globali
RW = 2.0  # peso casi 2021+ (prova 2.0 in una seconda submission)
Pk, _ = run(VAR, x_all, x_kag, RW)
sub = pd.DataFrame(Pk.reshape(len(Pk), -1), columns=out_cols)[list(y_train.columns)]
sub.insert(0, 'Run No', mp['Run No'].to_numpy())
ss = pd.read_csv(SS); norm = lambda c: c.lower().replace(' ', '_'); nm = {norm(c): c for c in ss.columns}
sub = sub.rename(columns=lambda c: nm.get(norm(c), c))  # nomi colonna come in sample_submission.csv
assert set(sub.columns) == set(ss.columns), set(sub.columns) ^ set(ss.columns)
sub = sub.set_index('Run No').loc[ss['Run No']].reset_index()[list(ss.columns)]
sub.to_csv('./Users/lucaag/data/submission_v12.csv', index=False)
print(sub.shape, sub.iloc[:, 1:].to_numpy().mean().round(3))

vesle_kjela ok
songa ok
byrte ok
haukeli ok
kjela ok
lio ok
vinje ok
tokke ok
hogga ok
(725, 2353) 0.462
